# Convolution Analysis · Example walkthrough

Use the same analysis engine, plots, uncertainty calculations and report exporter
as the desktop application, directly from Python. Choose a bundled example below
to load its CV, measured blank and saved parameter preset.

Run the cells in order. Each plot has its own figure. The last cell writes a new
report under `results/notebook/`; the input files and presets are left unchanged.

## Setup

From the repository root, activate the Python environment used to install the
application, then run:

```bash
python -m pip install .
python -m pip install jupyterlab
python -m jupyter lab
```

Open this notebook from `examples/` and choose that environment's Python kernel.
JupyterLab is an optional dependency;
see its [installation guide](https://jupyterlab.readthedocs.io/en/stable/getting_started/installation.html).
Keep this notebook inside the repository so it can locate the bundled datasets.

In [ ]:
from pathlib import Path
import sys

# Locate the repository when started from its root or from examples/.
working_directory = Path.cwd().resolve()
project_root = next(
    (folder for folder in (working_directory, *working_directory.parents)
     if (folder / "convolution_cv").is_dir() and (folder / "examples").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start this notebook from the repository root or examples/.")

# Use the source code belonging to these examples.
sys.path.insert(0, str(project_root))

%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from convolution_cv import (
    BACKGROUND_CV, __version__, analyze_cv, calculate_k_s, calculate_k_s_error,
    preview_preprocessing, recalculate_error_analysis,
)
from convolution_cv.ui.parameter_schema import load_preset, build_analysis_config
from convolution_cv.plotting import (
    draw_preprocessing, draw_processed_cv, draw_concentration_o,
    draw_concentration_r, draw_lgk,
)
from convolution_cv.formatting import format_e0, format_k_s
from convolution_cv.back_simulation import (
    run_back_simulation, comparison_view, draw_comparison,
)
from convolution_cv.reporting import export_analysis_result

print(f"Convolution Analysis {__version__}")

## 1. Choose an example and load its data

Each demo contains a CV, a measured blank and a JSON preset. The preset loader
resolves the blank's relative path beside the JSON file. In measured-blank mode,
the unused capacitance value is set to zero, as in the desktop application.

Set `CASE` below to one of the folder names in `case_files`. This single choice
selects the CV, blank, preset and report name for the entire walkthrough. After
changing it, rerun the selection cell and all following cells to refresh the
preview, results, uncertainty analysis, simulation and saved report.

For your own data, replace `cv_path` and `preset_path` with your CSV and a preset
saved by the application, and set `case_name` to the label you want in exports.

In [ ]:
CASE = "ferrocyanide"  # Choose any folder name listed in case_files below.
case_files = {
    "ferrocyanide": "10V",
    "ferrocene": "20V",
    "dioxygen": "50V",
    "tris_bipyridine_ruthenium": "50V",
    "tert_nitrobutane": "10V",
}
case_names = {
    "ferrocyanide": "Ferrocyanide",
    "ferrocene": "Ferrocene",
    "dioxygen": "Dioxygen",
    "tris_bipyridine_ruthenium": "Tris(2,2′-bipyridine)ruthenium",
    "tert_nitrobutane": "tert-Nitrobutane",
}
if CASE not in case_files:
    raise ValueError(f"Choose CASE from: {', '.join(case_files)}")

case_name = case_names[CASE]
case_folder = project_root / "examples" / CASE
cv_path = case_folder / f"{case_files[CASE]}.csv"
preset_path = case_folder / f"{case_files[CASE]}.json"

main_values, error_values = load_preset(str(preset_path))
if main_values["background_mode"] == BACKGROUND_CV:
    main_values["double_layer_capacity"] = "0"

print(f"Case: {case_name}")
print(f"CV: {cv_path.name}")
print(f"Preset: {preset_path.name}")
print(f"Blank: {Path(main_values['background_cv_path']).name}")

## 2. Choose the cycle and smoothing

`cv_cycles_n` is the **one-based cycle to analyze**, not a count of cycles to
combine. The initial selection comes from the chosen example's preset. All plotted
curves show the selected cycle; the engine retains the imported history needed
for the calculation.

The default SG smoothing window is **20 mV**, with polynomial order **2**. The
window is a potential span, not a number of points. Set `SMOOTHING_ENABLED` to
`False` for no smoothing. The measured blank is subtracted point by point before
smoothing the corrected CV; it is not smoothed separately.

After editing these settings, rerun this cell and the following cells to refresh
the preview and all derived results.

In [ ]:
SELECTED_CYCLE = int(main_values["cv_cycles_n"])
SMOOTHING_ENABLED = True
SMOOTHING_WINDOW_MV = 20.0
SMOOTHING_ORDER = 2

main_values.update(
    cv_cycles_n=str(SELECTED_CYCLE),
    smoothing_enabled=str(SMOOTHING_ENABLED),
    smoothing_window_mv=str(SMOOTHING_WINDOW_MV),
    smoothing_order=str(SMOOTHING_ORDER),
)
config = build_analysis_config(main_values)  # Uncertainty is configured in step 5.

display(pd.DataFrame([
    ("Selected cycle", config.cv_cycles_n, "one-based"),
    ("Electron number", config.n_e, ""),
    ("Electrode diameter", config.diameter, "cm"),
    ("Scan rate", config.scan_rate, "V/s"),
    ("Temperature", config.temperature, "K"),
    ("Diffusion coefficient O", config.diffusion_o, "cm²/s"),
    ("Diffusion coefficient R", config.diffusion_r, "cm²/s"),
], columns=["Parameter", "Value", "Unit"]).set_index("Parameter"))

## 3. Review preprocessing

The preview uses the same colors as the application: raw CV in brown, blank in
gray, background-corrected CV in translucent blue, and smoothed CV in dark blue.
Reported warnings are shown below the figure.

In [ ]:
def show_plot(drawer, *args, **kwargs):
    """Draw one full-size figure with the application's shared styling."""
    figure, axis = plt.subplots(figsize=(10, 6.5), layout="constrained")
    drawer(axis, *args, **kwargs)
    plt.show()
    plt.close(figure)


preview = preview_preprocessing(str(cv_path), config)
print(f"Imported samples: {len(preview.potential):,}")
print(f"Complete cycles: {len(preview.segments) // 2}")
if preview.smoothing_enabled:
    print(f"Effective SG window: {preview.smoothing_effective_mv:g} mV "
          f"({preview.smoothing_points} samples), order {preview.smoothing_order}")
else:
    print("Smoothing: off")

show_plot(draw_preprocessing, preview, selected_cycle=config.cv_cycles_n)
for warning in preview.warnings:
    print(f"Warning: {warning}")

## 4. Run the main analysis

The returned `result` contains full-precision arrays and parameters. Formatting
below only changes their presentation. E₀ and kₛ follow the application's
backward-scan calculation; the kinetic plot also shows the backward scan.

In [ ]:
result = analyze_cv(str(cv_path), config)
rate = calculate_k_s(result)

display(pd.DataFrame([
    ("Formal potential E₀", format_e0(result.e0)),
    ("Standard rate constant kₛ", format_k_s(rate.k_s)),
], columns=["Result", "Value"]).set_index("Result"))

for warning in result.warnings:
    print(f"Warning: {warning}")

In [ ]:
for drawer in (draw_processed_cv, draw_concentration_o, draw_concentration_r, draw_lgk):
    show_plot(drawer, result)

## 5. Add uncertainty analysis

The preset supplies current uncertainty in A, capacitance uncertainty in F,
diameter/diffusion uncertainties in percent, and the maximum accepted log₁₀
interval width. These are the example's saved settings, not universal values
for new experiments.

Edit `error_values` before building `error_config` if needed. The shared preset
adapter converts percentages to absolute uncertainties. Recalculation below
reuses the main result and returns a new result with uncertainty information.

In [ ]:
# Optional overrides, using the same field names as the preset:
# error_values["sigma_current"] = "1e-6"
# error_values["lgk_error_limit"] = "0.3"
display(pd.Series(error_values, name="Preset value").to_frame())

error_config = build_analysis_config(main_values, error_values)
result_with_error = recalculate_error_analysis(
    result,
    sigma_current=error_config.sigma_current,
    sigma_cdl=error_config.sigma_cdl,
    sigma_diameter=error_config.sigma_diameter,
    sigma_diffusion_o=error_config.sigma_diffusion_o,
    sigma_diffusion_r=error_config.sigma_diffusion_r,
    lgk_error_limit=error_config.lgk_error_limit,
)

display(pd.DataFrame([
    ("Formal potential E₀", format_e0(result_with_error.e0, result_with_error.error_e0)),
    ("Standard rate constant kₛ",
     format_k_s(calculate_k_s(result_with_error).k_s, calculate_k_s_error(result_with_error))),
], columns=["Result", "Value with uncertainty"]).set_index("Result"))

for warning in result_with_error.warnings:
    if warning not in result.warnings:
        print(f"Warning: {warning}")

Accepted samples are larger and opaque; other samples are translucent in the
same series color. Unbounded upper kinetic intervals reach the top of the plot.
The numeric sentinel used internally is not treated as a finite uncertainty.

In [ ]:
for drawer in (draw_concentration_o, draw_concentration_r, draw_lgk):
    show_plot(drawer, result_with_error, show_error=True, kinetics_scan="backward")

## 6. Compare a back simulation

Use extracted E₀ and kₛ with planar diffusion and the application's estimated
initial concentration and single starting species. The default model is
Butler–Volmer (`"BV"`) with α = 0.5. To use MHL, set `KINETIC_MODEL = "MHL"`
and supply a positive `LAMBDA_EV` in eV. α is used only for B-V; λ only for MHL.

Set `RUN_SIMULATION = False` to skip this optional step. If you change the main
analysis settings, rerun the main analysis and this section before saving.

In [ ]:
RUN_SIMULATION = True
KINETIC_MODEL = "BV"  # "BV" or "MHL"
ALPHA = 0.5
LAMBDA_EV = None  # For MHL, enter the reorganization energy in eV.

simulation = None
if RUN_SIMULATION:
    simulation = run_back_simulation(
        result_with_error,
        kinetic_model=KINETIC_MODEL,
        alpha=ALPHA,
        lambda_eV=LAMBDA_EV,
    )
    for warning in simulation.simulation.diagnostics["warnings"]:
        print(f"Warning: {warning}")

### Comparison smoothing

These independent display settings initially match the preprocessing settings.
SG smoothing is applied equally to the measured and simulated traces for the
comparison. It does not change the simulated current or fitted/extracted
parameters. Rerun this cell alone to change the view without rerunning the solver.

Both the unsmoothed and displayed comparison errors are reported below. Residuals
are experiment minus simulation; only the selected cycle is plotted.

In [ ]:
COMPARISON_SMOOTHING_ENABLED = SMOOTHING_ENABLED
COMPARISON_WINDOW_MV = SMOOTHING_WINDOW_MV
COMPARISON_ORDER = SMOOTHING_ORDER

comparison_smoothing = {
    "smoothing_enabled": COMPARISON_SMOOTHING_ENABLED,
    "smoothing_window_mv": COMPARISON_WINDOW_MV,
    "smoothing_order": COMPARISON_ORDER,
}
if simulation is not None:
    view = comparison_view(simulation, comparison_smoothing)
    show_plot(draw_comparison, simulation, view)
    show_plot(draw_comparison, simulation, view, residual=True)
    display(pd.DataFrame([
        (label, view[key]["selected_cycle"]["rmse_a"] * 1e6,
         view[key]["selected_cycle"]["max_absolute_error_a"] * 1e6)
        for label, key in (("Unsmoothed", "raw_metrics"), ("Displayed", "displayed_metrics"))
    ], columns=["Comparison", "RMSE (µA)", "Max. absolute error (µA)"]).set_index("Comparison"))
    for warning in view["warnings"]:
        print(f"Warning: {warning}")
else:
    print("Back simulation skipped.")

## 7. Save a complete report

This cell uses the same exporter as **Save Latest Results**. Every run creates a
new timestamped folder, including the HTML overview, full-precision data,
PNG/SVG/PDF figures, settings and input copies. If simulation was run, its data and
the current comparison view are included.

Open `Overview.html` from the printed folder in a browser; `FILE_GUIDE.md`
describes the saved files. The repository ignores generated `results/` files.

In [ ]:
output_root = project_root / "results" / "notebook"
report_folder = export_analysis_result(
    result_with_error,
    str(output_root),
    data_name=case_name,
    include_input_copy=True,
    include_error_figures=True,
    simulation=simulation,
    simulation_smoothing=comparison_smoothing,
)
print(f"Report folder: {report_folder}")
print(f"Open in a browser: {report_folder / 'Overview.html'}")

## Working with your own data

Use a CSV with `Potential/V` and `Current/A` columns and a matching parameter
preset saved from Configure. With measured-blank subtraction, the blank must meet
the application's point-by-point alignment requirements. Update the paths in
step 1, review the parameters and uncertainties, and run all following cells.

For conventions and output definitions, see the
[user guide](../docs/USER_GUIDE.md),
[calculation method](../docs/DATA_ANALYSIS_METHOD.md) and
[back-simulation guide](../docs/BACK_SIMULATION.md).